# Family A / Run 4 training -- baseline vs. auxiliary, two seeds. Exploratory, DO NOT SUBMIT.

Requires the pool and gold cache notebooks to have already run in this session (or an equivalent /kaggle/working/family_a_src + cache directories).


In [ ]:
from __future__ import annotations
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback')
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device('cpu')]
print(f'devices: {[str(d) for d in DEVS]}')
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for depth1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [depth1] + sorted((p for p in depth1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError(f'competition mount not found (cwd {Path.cwd()}); expected a directory holding test.csv and test_series/')

def find_dinov2(variant='small'):
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 'config.json' in files and 'dinov2' in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None
LABEL_COLS = TARGETS + [t + '__conf' for t in TARGETS]

class LabelSourceError(RuntimeError):
    pass

def find_label_table():
    base = Path('/kaggle/input')
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
            cands += [Path(root) / f for f in files if f.startswith('report_labels') and f.endswith('.csv')]
    cands += [p for p in (Path('data/derived/report_labels_v2.csv'),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if 'StudyInstanceUID' in head.columns and all((t in head.columns for t in TARGETS)):
            return c
    return None

def label_mount_attached():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return False
    return any(('label' in p.name.lower() for p in base.iterdir() if p.is_dir()))

def read_labels(train_df):
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    lab['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    lab = lab.set_index('StudyInstanceUID')
    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError('LABEL SOURCE: a label dataset is mounted but no usable table was found in it. Falling back to the lexicon here would train on the weaker labels and say so only in a log line, so the run stops instead.')
        log(f'LABEL SOURCE: lexicon, {n} studies (no table mounted)')
        return lab
    tab = pd.read_csv(src).set_index('StudyInstanceUID')
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(f'LABEL SOURCE: {src} is missing {len(missing)} expected columns (first: {missing[0]!r}). Refusing to fall back silently.')
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(f'LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.')
    log(f'LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, lexicon for the remaining {n - len(hit)}')
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab
ROOT = find_root()
log(f'input root: {ROOT}')
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')

In [ ]:
import sys
from pathlib import Path
_src = Path('/kaggle/working/family_a_src')
_src.mkdir(parents=True, exist_ok=True)
(_src / 'contract.py').write_text('"""Tensor/metadata contract shared by every Family A component.\n\nMirrors the already-deployed, already-tested V13 six-slot layout (SLOTS, GROUP,\nIMG, CROP_MM, SLOT_PRIOR_TABLE) so a raw-slot-tensor cache built by reusing V13\'s\nown pick_slots/order_slices/read_slot cells (see build_pool_notebook.py) is\ndirectly consumable here without a second, independently-written and unvalidated\nDICOM decoder. Family A\'s model and supervision are new; the pixel pipeline is\nreused, not reinvented.\n"""\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\n# name, plane, fluid-sensitive (None = don\'t care), fatsat\nSLOTS = [\n    (\'SAG_FLUID_FS\', \'Sagittal\', True, True),\n    (\'COR_FLUID_FS\', \'Coronal\', True, True),\n    (\'AX_FLUID_FS\', \'Axial\', True, True),\n    (\'SAG_FLUID_NOFS\', \'Sagittal\', True, False),\n    (\'COR_T1\', \'Coronal\', False, False),\n    (\'SAG_T1\', \'Sagittal\', False, False),\n]\nN_SLOT = len(SLOTS)\nGROUP = 3        # adjacent slices per window, fed as pseudo-RGB channels\nIMG = 336        # per-slot tile side, pixels\nCROP_MM = 130.0\n\nSLOT_PRIOR_TABLE = {\n    \'ACL\': (0, 3, 5), \'MCL\': (1, 4), \'Medial Meniscus\': (0, 1, 3, 4),\n    \'Lateral Meniscus\': (0, 1, 3, 4), \'Medial OA\': (1, 4, 5), \'Lateral OA\': (1, 4, 5),\n    \'PF OA\': (0, 2, 5), \'Effusion\': (0, 2), \'Synovitis\': (0, 2), "Baker\'s": (0,),\n    \'Contusion\': (0, 1, 2), \'Fracture\': (0, 1, 2, 4, 5),\n}\nSLOT_PRIOR_STRENGTH = 0.55\n', encoding='utf-8')
(_src / 'labels.py').write_text('"""Label construction: expert gold labels are the primary objective; a capped,\nevidence-gated auxiliary target from report-derived soft labels is optional.\n\nCorrections applied per review, encoded here rather than left as prose policy:\n\n  * Report-silent cells (raw value == 0.5) are EXCLUDED from the auxiliary loss\n    (mask weight 0), never filled in as a hard negative. "Masked" and "filled as\n    negative" are different treatments; Run 4 uses the former exclusively.\n  * A target gets nonzero auxiliary weight only if its transfer_audit.py\n    addressed-only AUC bootstrap CI lower bound exceeds 0.5 AND its Brier skill\n    score (vs. the prevalence-only baseline) is positive. AUC alone is not\n    sufficient: MCL has AUC 0.958 but skill only 0.11 (near-chance calibration\n    despite near-perfect ranking); Lateral OA/Synovitis/Contusion have AUC CI\n    lower bounds above 0.5 but NEGATIVE skill and are excluded despite passing\n    on AUC alone.\n  * A target flagged small_sample_warning (fewer than 10 studies in the smaller\n    addressed class) has its weight halved rather than trusted at face value.\n  * This policy is derived from the same 58 gold studies Run 4 later scores\n    against. Any Run 4 result on that same 58-study cohort is exploratory, not\n    independent confirmation of the policy that selected it.\n"""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom contract import TARGETS, UID\n\nMAX_AUX_WEIGHT = 0.3\nSMALL_SAMPLE_WEIGHT_MULTIPLIER = 0.5\nSILENT_VALUE = 0.5\n\n\ndef load_transfer_policy(transfer_audit_report_path):\n    """Derive {target: {\'weight\': float, \'use_aux\': bool, \'reason\': str}} from a\n    transfer_audit.py report. Every decision carries its numeric justification\n    so the policy is auditable, not a hand-picked list."""\n    report = json.loads(Path(transfer_audit_report_path).read_text(encoding=\'utf-8\'))\n    policy = {}\n    for t in TARGETS:\n        d = report[\'targets\'][t]\n        auc_ci = d[\'auc_addressed_only\'][\'ci95\']\n        skill = d[\'brier_addressed_only\'][\'skill_score\']\n        if auc_ci is None or skill is None:\n            policy[t] = {\'weight\': 0.0, \'use_aux\': False,\n                         \'reason\': \'insufficient evidence: AUC CI or Brier skill unavailable at this n\'}\n            continue\n        ci_lower = auc_ci[0]\n        if ci_lower <= 0.5 or skill <= 0.0:\n            policy[t] = {\'weight\': 0.0, \'use_aux\': False,\n                         \'reason\': f\'gate failed: AUC CI lower={ci_lower:.3f}, skill={skill:.3f}\'}\n            continue\n        weight = min(skill, MAX_AUX_WEIGHT)\n        halved = bool(d[\'small_sample_warning\'])\n        if halved:\n            weight *= SMALL_SAMPLE_WEIGHT_MULTIPLIER\n        policy[t] = {\'weight\': float(weight), \'use_aux\': True,\n                     \'reason\': f\'gate passed: AUC CI lower={ci_lower:.3f}, skill={skill:.3f}\'\n                               + (\', weight halved (small sample)\' if halved else \'\')}\n    return policy\n\n\ndef zero_policy():\n    """The baseline-arm policy: auxiliary loss disabled for every target."""\n    return {t: {\'weight\': 0.0, \'use_aux\': False, \'reason\': \'baseline arm: auxiliary loss disabled\'}\n            for t in TARGETS}\n\n\ndef build_supervision(ids, gold_labels, report_source, policy, silent_value=SILENT_VALUE):\n    """Return (y_expert, expert_mask, y_aux, aux_mask, aux_weight) aligned to `ids`.\n\n    y_expert/expert_mask: expert gold labels where the study is in the gold\n    cohort, NaN/False elsewhere. This is the only signal Run 4 evaluates against.\n\n    y_aux/aux_mask: report-derived soft value, masked out (never filled as a\n    negative) wherever the report was silent OR the target failed the transfer\n    policy gate. Fold-safety (excluding validation-fold rows from any loss) is\n    the training loop\'s responsibility, not this function\'s -- these arrays are\n    fold-agnostic raw supervision.\n\n    aux_weight: one capped scalar per target from the policy, constant across\n    studies.\n    """\n    ids = list(ids)\n    n = len(ids)\n    y_expert = np.full((n, len(TARGETS)), np.nan)\n    expert_mask = np.zeros((n, len(TARGETS)), dtype=bool)\n    if gold_labels is not None and len(gold_labels):\n        gold = gold_labels.set_index(UID) if UID in gold_labels.columns else gold_labels\n        if gold.index.duplicated().any():\n            raise ValueError(\'Duplicate study IDs in gold labels\')\n        for i, uid in enumerate(ids):\n            if uid in gold.index:\n                row = gold.loc[uid, TARGETS].to_numpy(float)\n                y_expert[i] = row\n                expert_mask[i] = np.isfinite(row)\n\n    report = report_source.set_index(UID) if UID in report_source.columns else report_source\n    if report.index.duplicated().any():\n        raise ValueError(\'Duplicate study IDs in report source\')\n    missing = [u for u in ids if u not in report.index]\n    if missing:\n        raise ValueError(f\'{len(missing)} studies missing from report source, e.g. {missing[:3]}\')\n    raw = report.loc[ids, TARGETS].to_numpy(float)\n    if not (np.isfinite(raw) & (raw >= 0) & (raw <= 1)).all():\n        raise ValueError(\'Report-derived values must be finite probabilities in [0, 1]\')\n\n    addressed = raw != silent_value\n    use_aux = np.array([policy.get(t, {}).get(\'use_aux\', False) for t in TARGETS])\n    aux_mask = addressed & use_aux[None, :]\n    y_aux = np.where(aux_mask, raw, np.nan)\n    aux_weight = np.array([policy.get(t, {}).get(\'weight\', 0.0) for t in TARGETS], dtype=float)\n    return y_expert, expert_mask, y_aux, aux_mask, aux_weight\n', encoding='utf-8')
(_src / 'model.py').write_text('"""Family A: six-slot 2.5D multiple-instance model with target-query attention.\n\nThe pooling head\'s math matches the deployed V13 SlotHead exactly (proj -> add\nslot embedding -> per-target query attention over slots, masked-softmax, output\nprojection, optional slot-anatomy prior bias) since that mechanism is already\nvalidated in production; what\'s new is the encoder is trainable here (unfrozen\nlast blocks, per the plan\'s Phase A1) rather than used only as a frozen feature\nsource the way the failed specialist experiments did, and the model is trained\ndirectly (with capped auxiliary supervision), not as a post-hoc residual\ncorrection on top of another model\'s output.\n\nEncoders are pluggable so architecture correctness can be verified on CPU with\nTinyCNNEncoder (no external weights, no GPU) before ever touching a real image\nor the transformers/DINOv2 dependency, which Dinov2Encoder needs and which only\nmatters on the actual training device.\n"""\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom contract import GROUP, IMG, N_SLOT, SLOT_PRIOR_STRENGTH, SLOT_PRIOR_TABLE, SLOTS, TARGETS\n\n\nclass TargetQueryHead(nn.Module):\n    """One learned query per target attends over the N_SLOT slot tokens."""\n\n    def __init__(self, dim, n_slot=N_SLOT, n_out=len(TARGETS), hidden=256, dropout=0.2, use_prior=True):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())\n        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)\n        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(hidden, n_out)\n        self.hidden = hidden\n        self.use_prior = use_prior and n_slot == len(SLOTS) and n_out == len(TARGETS)\n        prior = torch.zeros(n_out, n_slot)\n        if self.use_prior:\n            for t, slots in SLOT_PRIOR_TABLE.items():\n                if t in TARGETS:\n                    prior[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH\n        self.register_buffer(\'slot_prior\', prior)\n\n    def forward(self, tokens, mask):\n        """tokens: (B, n_slot, dim). mask: (B, n_slot) in {0, 1}, 1 = slot present.\n        Returns (B, n_out) logits and (B, n_out, n_slot) attention weights."""\n        h = self.proj(tokens) + self.slot_emb\n        att = torch.einsum(\'bsh,oh->bos\', h, self.query) / self.hidden ** 0.5\n        if self.use_prior:\n            att = att + self.slot_prior.unsqueeze(0)\n        if (mask.sum(dim=1) == 0).any():\n            raise ValueError(\'At least one slot must be present per study; an all-missing study cannot be scored\')\n        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)\n        ctx = self.drop(torch.einsum(\'bos,bsh->boh\', att, h))\n        logits = (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias\n        return logits, att.detach()\n\n\nclass TinyCNNEncoder(nn.Module):\n    """CPU-only stand-in encoder for architecture/shape/gradient-flow testing.\n    Not a modeling contribution -- swap in Dinov2Encoder for real training."""\n\n    def __init__(self, out_dim=64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(GROUP, 16, 5, stride=4, padding=2), nn.GELU(),\n            nn.Conv2d(16, 32, 5, stride=4, padding=2), nn.GELU(),\n            nn.AdaptiveAvgPool2d(1),\n        )\n        self.proj = nn.Linear(32, out_dim)\n        self.out_dim = out_dim\n\n    def forward(self, x):\n        """x: (N, GROUP, IMG, IMG) float in [0, 1]. Returns (N, out_dim)."""\n        return self.proj(self.net(x).flatten(1))\n\n\nclass Dinov2Encoder(nn.Module):\n    """Wraps a HuggingFace DINOv2 backbone with partial unfreezing, matching the\n    already-tested V13 build_model() unfreeze/normalize contract exactly, so a\n    checkpoint trained here is not fighting a second, different preprocessing\n    convention. Only imported/instantiated on the GPU training device; the\n    `transformers` dependency and pretrained weights are not needed for the CPU\n    smoke test."""\n\n    def __init__(self, source_path, unfreeze_last=2, pool=\'cls_mean\'):\n        super().__init__()\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(str(source_path))\n        n_layer = len(backbone.encoder.layer)\n        for p in backbone.parameters():\n            p.requires_grad = False\n        for blk in backbone.encoder.layer[max(0, n_layer - unfreeze_last):]:\n            for p in blk.parameters():\n                p.requires_grad = True\n        for p in backbone.layernorm.parameters():\n            p.requires_grad = True\n        self.backbone = backbone\n        self.pool = pool\n        parts = {\'cls_mean\': 2, \'cls_mean_focal\': 3}[pool]\n        self.out_dim = backbone.config.hidden_size * parts\n        self.register_buffer(\'mean\', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer(\'std\', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, x):\n        x = (x - self.mean) / self.std\n        out = self.backbone(pixel_values=x).last_hidden_state\n        patch = out[:, 1:]\n        parts = [out[:, 0], patch.mean(1)]\n        if self.pool == \'cls_mean_focal\':\n            k = max(1, patch.shape[1] // 8)\n            parts.append(patch.topk(k, dim=1).values.mean(1))\n        return torch.cat(parts, dim=1)\n\n\nclass GeneralistModel(nn.Module):\n    def __init__(self, encoder, n_slot=N_SLOT, n_out=len(TARGETS), head_hidden=256, dropout=0.2, use_prior=True):\n        super().__init__()\n        self.encoder = encoder\n        self.head = TargetQueryHead(encoder.out_dim, n_slot, n_out, head_hidden, dropout, use_prior)\n\n    def forward(self, imgs, mask, img_size=None):\n        """imgs: (B, n_slot, GROUP, H, W) uint8 or float. mask: (B, n_slot)."""\n        b, s = imgs.shape[:2]\n        x = imgs.reshape(b * s, *imgs.shape[2:]).float()\n        if x.max() > 1.5:  # tensor arrived as 0-255 uint8-range values\n            x = x / 255.0\n        if img_size is not None and img_size != x.shape[-1]:\n            x = F.interpolate(x, size=(img_size, img_size), mode=\'bilinear\', align_corners=False)\n        tokens = self.encoder(x).reshape(b, s, -1)\n        return self.head(tokens, mask)\n\n\ndef build_smoke_model(hidden=32, out_dim=16):\n    return GeneralistModel(TinyCNNEncoder(out_dim=out_dim), head_hidden=hidden)\n', encoding='utf-8')
(_src / 'folds.py').write_text('"""Deterministic grouped fold assignment.\n\nThe repository\'s own PatientID audit found singleton groups in the available\nmetadata (no verified repeat-patient linkage), so grouping by patient identity\ncannot currently be claimed here. This groups by StudyInstanceUID -- i.e. every\nstudy is its own group -- which is a weaker guarantee than true patient\nseparation and is labeled as such in every receipt this module contributes to,\nrather than silently presented as patient-safe splitting.\n"""\nimport numpy as np\n\n\ndef assign_folds(ids, k=5, seed=1400):\n    if k < 2:\n        raise ValueError(\'Need at least 2 folds\')\n    ids = list(ids)\n    n = len(ids)\n    if n < k:\n        raise ValueError(f\'Need at least as many studies ({n}) as folds ({k})\')\n    if len(set(ids)) != n:\n        raise ValueError(\'Duplicate study IDs cannot be assigned to folds\')\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(n)\n    fold_of = np.empty(n, dtype=int)\n    fold_of[order] = np.arange(n) % k\n    return {\n        \'fold_assignment\': dict(zip(ids, fold_of.tolist())),\n        \'k\': k, \'seed\': seed, \'n_studies\': n,\n        \'grouping_caveat\': (\'Grouped by StudyInstanceUID, one study per group. Patient-level \'\n                            \'grouping was not used: the repository\\\'s metadata audit found only \'\n                            \'singleton patient groups, so patient-safe separation is unverified.\'),\n    }\n', encoding='utf-8')
(_src / 'losses.py').write_text('"""Masked BCE for expert labels, plus an optional capped, masked auxiliary term.\n\nBoth losses are computed with logits (not probabilities) via\nbinary_cross_entropy_with_logits for numerical stability, and both are averaged\nonly over the cells their own mask marks valid -- a study/target cell that is\nNaN in y (no expert label, or report-silent for the auxiliary target) never\nenters either loss, it is not treated as a zero.\n"""\nimport torch\nimport torch.nn.functional as F\n\n\ndef masked_bce(logits, y, mask):\n    """logits, y, mask: (B, T). Returns a scalar; 0.0 (not NaN) if mask is empty\n    so a batch with no expert-labeled rows for a given fold split doesn\'t produce\n    a NaN gradient step."""\n    if mask.sum() == 0:\n        return logits.new_zeros(())\n    y_safe = torch.where(mask, y, torch.zeros_like(y))\n    per_cell = F.binary_cross_entropy_with_logits(logits, y_safe, reduction=\'none\')\n    return (per_cell * mask.float()).sum() / mask.float().sum()\n\n\ndef combined_loss(logits, y_expert, expert_mask, y_aux, aux_mask, aux_weight):\n    """aux_weight: (T,) capped per-target scalar, broadcast across the batch and\n    multiplied into the per-cell auxiliary mask before averaging, so a target\n    with weight 0 (rejected by the transfer-gate policy, or the baseline arm\n    where every weight is 0) contributes exactly nothing -- not a small nonzero\n    push -- to the total loss."""\n    expert_term = masked_bce(logits, y_expert, expert_mask)\n    if aux_weight is None or float(aux_weight.abs().sum()) == 0.0:\n        return expert_term, {\'expert\': float(expert_term.detach()), \'aux\': 0.0}\n    y_safe = torch.where(aux_mask, y_aux, torch.zeros_like(y_aux))\n    per_cell = F.binary_cross_entropy_with_logits(logits, y_safe, reduction=\'none\')\n    weighted_mask = aux_mask.float() * aux_weight.unsqueeze(0)\n    denom = weighted_mask.sum()\n    aux_term = (per_cell * weighted_mask).sum() / denom if denom > 0 else logits.new_zeros(())\n    total = expert_term + aux_term\n    return total, {\'expert\': float(expert_term.detach()), \'aux\': float(aux_term.detach())}\n', encoding='utf-8')
(_src / 'cache.py').write_text('"""Chunked, checksum-backed slot-tensor cache: one file per study, never one\ngiant fragile array, per the plan\'s cache design. Real cache contents are built\non the GPU device by build_pool_notebook.py (which reuses V13\'s own\npick_slots/order_slices/read_slot cells against real DICOMs); synth() here\nbuilds a structurally identical fake cache from random tensors so every other\nFamily A component can be exercised on CPU without any competition data.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom contract import GROUP, IMG, N_SLOT, UID\n\n\ndef _study_path(cache_dir, uid):\n    if \'/\' in uid or \'\\\\\' in uid or uid in (\'.\', \'..\'):\n        raise ValueError(\'Unsafe study identifier\')\n    return Path(cache_dir) / f\'{uid}.npz\'\n\n\ndef write_study(cache_dir, uid, imgs, mask):\n    imgs = np.asarray(imgs)\n    mask = np.asarray(mask)\n    if imgs.shape != (N_SLOT, GROUP, IMG, IMG):\n        raise ValueError(f\'imgs must be shape {(N_SLOT, GROUP, IMG, IMG)}, got {imgs.shape}\')\n    if mask.shape != (N_SLOT,):\n        raise ValueError(f\'mask must be shape {(N_SLOT,)}, got {mask.shape}\')\n    if not mask.any():\n        raise ValueError(f\'{uid}: at least one slot must be present\')\n    if imgs.dtype != np.uint8:\n        raise ValueError(\'imgs must be uint8 (percentile-normalized, 0-255)\')\n    path = _study_path(cache_dir, uid)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, imgs=imgs, mask=mask.astype(np.float32))\n    return path\n\n\ndef build_manifest(cache_dir, ids):\n    cache_dir = Path(cache_dir)\n    entries = {}\n    missing = []\n    for uid in ids:\n        path = _study_path(cache_dir, uid)\n        if not path.is_file():\n            missing.append(uid)\n            continue\n        with np.load(path) as f:\n            imgs, mask = f[\'imgs\'], f[\'mask\']\n        if imgs.shape != (N_SLOT, GROUP, IMG, IMG) or mask.shape != (N_SLOT,):\n            raise ValueError(f\'{uid}: cached tensor does not match the current contract shape\')\n        entries[uid] = {\'path\': path.name, \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                        \'n_slots_present\': int(mask.sum())}\n    if missing:\n        raise FileNotFoundError(f\'{len(missing)} studies missing from cache, e.g. {missing[:3]}\')\n    manifest = {\'contract\': {\'n_slot\': N_SLOT, \'group\': GROUP, \'img\': IMG},\n                \'studies\': entries, \'n_studies\': len(entries)}\n    (cache_dir / \'manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    return manifest\n\n\ndef load_manifest(cache_dir):\n    return json.loads((Path(cache_dir) / \'manifest.json\').read_text(encoding=\'utf-8\'))\n\n\ndef load_study(cache_dir, uid):\n    with np.load(_study_path(cache_dir, uid)) as f:\n        return f[\'imgs\'], f[\'mask\']\n\n\ndef load_batch(cache_dir, ids):\n    imgs, masks = [], []\n    for uid in ids:\n        i, m = load_study(cache_dir, uid)\n        imgs.append(i)\n        masks.append(m)\n    return np.stack(imgs), np.stack(masks)\n\n\ndef synth(cache_dir, ids, seed=1400, min_slots_present=3):\n    """Build a structurally valid fake cache for the CPU smoke test. Not real\n    image data; exercises every downstream shape/mask/manifest code path."""\n    rng = np.random.default_rng(seed)\n    for uid in ids:\n        n_present = rng.integers(min_slots_present, N_SLOT + 1)\n        mask = np.zeros(N_SLOT, dtype=bool)\n        mask[rng.choice(N_SLOT, size=n_present, replace=False)] = True\n        imgs = rng.integers(0, 256, size=(N_SLOT, GROUP, IMG, IMG), dtype=np.uint8)\n        write_study(cache_dir, uid, imgs, mask)\n    return build_manifest(cache_dir, ids)\n', encoding='utf-8')
(_src / 'engine.py').write_text('"""Training loop: deterministic seeding, grouped folds, checkpoint resume,\nper-arm (baseline / auxiliary) supervision, OOF export, and a receipt for every\nrun. Loss is computed on the training split only; validation-fold rows are\nscored with a bare forward pass (no grad) purely to produce the OOF prediction\nfor that row -- this is also why no special-casing is needed to keep a fold\'s\nheld-out expert label from leaking through the auxiliary term: the auxiliary\nloss is only ever summed over training-split rows in the first place.\n"""\nimport contextlib\nimport hashlib\nimport json\nimport os\nimport random\nimport subprocess\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom compare_oof import aucs\nfrom contract import TARGETS, UID\nfrom folds import assign_folds\nfrom labels import build_supervision\nfrom losses import combined_loss\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef git_commit():\n    try:\n        return subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], capture_output=True, text=True,\n                              cwd=Path(__file__).resolve().parent, check=True).stdout.strip()\n    except Exception:\n        return None\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef atomic_save(path, state):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    torch.save(state, tmp)\n    os.replace(tmp, path)\n\n\ndef write_receipt(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(data, f, indent=2, default=str)\n        f.write(\'\\n\')\n\n\ndef rng_state():\n    return {\'python\': random.getstate(), \'numpy\': np.random.get_state(), \'torch\': torch.get_rng_state()}\n\n\ndef restore_rng_state(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'])\n\n\ndef _batches(idx, batch_size, rng):\n    idx = idx.copy()\n    rng.shuffle(idx)\n    for start in range(0, len(idx), batch_size):\n        yield idx[start:start + batch_size]\n\n\ndef _predict(model, imgs_t, masks_t, idx):\n    model.eval()\n    with torch.no_grad():\n        idx_arr = np.array(idx)\n        logits, attention = model(imgs_t[idx_arr], masks_t[idx_arr])\n        return torch.sigmoid(logits).numpy(), attention.numpy()\n\n\ndef _val_macro_auc(probs, y_expert, expert_mask, val_idx):\n    """Macro AUC over whichever val-fold rows actually carry an expert label\n    (report-only pool studies in the validation split contribute nothing here,\n    same as they contribute nothing to the final scored OOF)."""\n    val_idx = np.array(val_idx)\n    gold_rows = expert_mask[val_idx].any(axis=1)\n    if gold_rows.sum() < 4:\n        return None  # too few gold rows in this fold\'s validation split to trust a per-epoch AUC\n    y = np.where(expert_mask[val_idx][gold_rows], y_expert[val_idx][gold_rows], np.nan)\n    per_target = aucs(y, probs[gold_rows])\n    valid = per_target[np.isfinite(per_target)]\n    return float(valid.mean()) if len(valid) else None\n\n\ndef train_one_fold(model_fn, imgs, masks, y_expert, expert_mask, y_aux, aux_mask, aux_weight,\n                    train_idx, val_idx, epochs, lr, seed, checkpoint_path, batch_size=8, resume=True,\n                    patience=None):\n    """Checkpoints every epoch (for resume) and separately tracks the best-so-far\n    epoch by validation macro AUC on this fold\'s gold rows, so a long run doesn\'t\n    have to guess the right epoch count in advance: OOF predictions and the final\n    receipt both come from the best epoch, not whichever epoch happened to be\n    last when training stopped. `patience`: stop early after this many epochs\n    with no improvement over the best (None = always run the full `epochs`)."""\n    device = torch.device(\'cpu\')\n    set_seed(seed)\n    model = model_fn().to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)\n    start_epoch = 0\n    stopping_reason = \'completed_all_epochs\'\n    checkpoint_path = Path(checkpoint_path)\n    best_path = checkpoint_path.with_suffix(\'.best.pt\')\n    best_epoch, best_val_auc, epochs_since_improvement = None, -float(\'inf\'), 0\n    if resume and checkpoint_path.is_file():\n        state = torch.load(checkpoint_path, map_location=device, weights_only=False)\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        restore_rng_state(state[\'rng_state\'])\n        start_epoch = state[\'epoch\'] + 1\n        best_epoch = state.get(\'best_epoch\')\n        best_val_auc = state.get(\'best_val_auc\', -float(\'inf\'))\n        epochs_since_improvement = state.get(\'epochs_since_improvement\', 0)\n\n    imgs_t = torch.from_numpy(imgs)\n    masks_t = torch.from_numpy(masks).float()\n    y_expert_t = torch.from_numpy(np.nan_to_num(y_expert, nan=0.0)).float()\n    expert_mask_t = torch.from_numpy(expert_mask)\n    y_aux_t = torch.from_numpy(np.nan_to_num(y_aux, nan=0.0)).float()\n    aux_mask_t = torch.from_numpy(aux_mask)\n    aux_weight_t = torch.from_numpy(aux_weight).float()\n\n    batch_rng = np.random.default_rng(seed + 1)\n    history = []\n    for epoch in range(start_epoch, epochs):\n        model.train()\n        epoch_losses = []\n        for batch in _batches(np.array(train_idx), batch_size, batch_rng):\n            has_expert = bool(expert_mask_t[batch].any())\n            has_aux = bool(aux_mask_t[batch].any()) and float(aux_weight_t.abs().sum()) > 0\n            if not (has_expert or has_aux):\n                # A batch that is entirely report-only pool studies under the\n                # baseline (zero-weight) policy has no supervised cell at all --\n                # skip it rather than backward() through a graph-disconnected\n                # zero loss, which raises rather than silently no-op-ing.\n                epoch_losses.append({\'expert\': 0.0, \'aux\': 0.0})\n                continue\n            optimizer.zero_grad()\n            logits, _ = model(imgs_t[batch], masks_t[batch])\n            loss, parts = combined_loss(logits, y_expert_t[batch], expert_mask_t[batch],\n                                        y_aux_t[batch], aux_mask_t[batch], aux_weight_t)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f\'non-finite loss at epoch {epoch}: {parts}\')\n            loss.backward()\n            optimizer.step()\n            epoch_losses.append(parts)\n\n        val_probs, _ = _predict(model, imgs_t, masks_t, val_idx)\n        val_auc = _val_macro_auc(val_probs, y_expert, expert_mask, val_idx)\n        improved = val_auc is not None and val_auc > best_val_auc\n        if improved:\n            best_val_auc, best_epoch, epochs_since_improvement = val_auc, epoch, 0\n            atomic_save(best_path, {\'model\': model.state_dict(), \'epoch\': epoch, \'val_macro_auc\': val_auc})\n        else:\n            epochs_since_improvement += 1\n        history.append({\'epoch\': epoch,\n                        \'mean_expert_loss\': float(np.mean([p[\'expert\'] for p in epoch_losses])),\n                        \'mean_aux_loss\': float(np.mean([p[\'aux\'] for p in epoch_losses])),\n                        \'val_macro_auc\': val_auc, \'improved\': improved})\n        atomic_save(checkpoint_path, {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                                      \'epoch\': epoch, \'rng_state\': rng_state(), \'seed\': seed,\n                                      \'best_epoch\': best_epoch, \'best_val_auc\': best_val_auc,\n                                      \'epochs_since_improvement\': epochs_since_improvement})\n        if patience is not None and epochs_since_improvement >= patience:\n            stopping_reason = f\'early_stop: no improvement for {patience} epochs\'\n            break\n\n    if best_epoch is None:\n        # never had >=4 gold rows to score in this fold\'s validation split, or no\n        # epoch ever beat -inf (e.g. epochs=0): fall back to the last trained epoch.\n        stopping_reason = \'no_valid_epoch_auc_available; used final epoch weights\'\n        probs, attention = _predict(model, imgs_t, masks_t, val_idx)\n    else:\n        best_state = torch.load(best_path, map_location=device, weights_only=False)\n        model.load_state_dict(best_state[\'model\'])\n        probs, attention = _predict(model, imgs_t, masks_t, val_idx)\n    return probs, history, stopping_reason, attention, best_epoch, best_val_auc\n\n\ndef run_arm(arm_name, cache_dir, ids, gold_ids, gold_labels, report_source, policy, out_dir,\n            model_fn, k=5, epochs=3, lr=1e-3, seed=1400, batch_size=8, resume=True, fold_seed=1400,\n            patience=None):\n    """Trains one arm (\'baseline\' has an all-zero policy; \'auxiliary\' uses a\n    transfer-gated policy) across k grouped folds and writes an OOF prediction\n    file plus a receipt. Only studies in `gold_ids` ever contribute an expert-BCE\n    term or an OOF row scored against ground truth; report-only pool studies\n    still participate in training (auxiliary term only) but are not evaluated."""\n    from cache import load_batch\n\n    start = time.time()\n    out_dir = Path(out_dir)\n    imgs, masks = load_batch(cache_dir, ids)\n    y_expert, expert_mask, y_aux, aux_mask, aux_weight = build_supervision(\n        ids, gold_labels, report_source, policy)\n\n    fold_info = assign_folds(ids, k=k, seed=fold_seed)\n    fold_of = np.array([fold_info[\'fold_assignment\'][u] for u in ids])\n\n    oof = np.full((len(ids), len(TARGETS)), np.nan)\n    oof_covered = np.zeros(len(ids), dtype=bool)\n    fold_histories = {}\n    checkpoints = {}\n    for fold in range(k):\n        val_idx = np.flatnonzero(fold_of == fold)\n        train_idx = np.flatnonzero(fold_of != fold)\n        if len(val_idx) == 0 or len(train_idx) == 0:\n            raise ValueError(f\'fold {fold} has an empty split; reduce k or add studies\')\n        ckpt = out_dir / \'checkpoints\' / f\'{arm_name}_fold{fold}.pt\'\n        probs, history, reason, _, best_epoch, best_val_auc = train_one_fold(\n            model_fn, imgs, masks, y_expert, expert_mask, y_aux, aux_mask, aux_weight,\n            train_idx, val_idx, epochs, lr, seed, ckpt, batch_size, resume, patience)\n        oof[val_idx] = probs\n        oof_covered[val_idx] = True\n        fold_histories[fold] = history\n        checkpoints[str(fold)] = {\n            \'path\': str(ckpt), \'sha256\': digest_file(ckpt), \'stopping_reason\': reason,\n            \'best_epoch\': best_epoch, \'best_val_macro_auc\': best_val_auc,\n            \'epochs_run\': len(history), \'epochs_requested\': epochs,\n        }\n\n    if not oof_covered.all():\n        raise RuntimeError(\'Incomplete OOF coverage; refusing to write a partial prediction file\')\n\n    gold_only = [i for i, u in enumerate(ids) if u in set(gold_ids)]\n    frame = pd.DataFrame(oof[gold_only], columns=TARGETS)\n    frame.insert(0, UID, [ids[i] for i in gold_only])\n    oof_path = out_dir / f\'{arm_name}_oof.csv\'\n    oof_path.parent.mkdir(parents=True, exist_ok=True)\n    with oof_path.open(\'x\', encoding=\'utf-8\') as f:\n        frame.to_csv(f, index=False)\n\n    best_epochs = [c[\'best_epoch\'] for c in checkpoints.values() if c[\'best_epoch\'] is not None]\n    receipt = {\n        \'arm\': arm_name, \'git_commit\': git_commit(), \'seed\': seed, \'fold_seed\': fold_seed,\n        \'k\': k, \'epochs_requested\': epochs, \'patience\': patience, \'lr\': lr, \'batch_size\': batch_size,\n        \'n_studies_total\': len(ids), \'n_studies_scored\': len(gold_only),\n        \'policy\': policy, \'fold_grouping_caveat\': fold_info[\'grouping_caveat\'],\n        \'checkpoints\': checkpoints, \'fold_histories\': fold_histories,\n        \'best_epoch_by_fold\': {k_: c[\'best_epoch\'] for k_, c in checkpoints.items()},\n        \'best_epoch_summary\': (\n            f\'range {min(best_epochs)}-{max(best_epochs)} across {len(best_epochs)}/{k} folds; \'\n            \'if this sits well below epochs_requested, more epochs were not being used -- raising \'\n            \'epochs_requested further is unlikely to help without also addressing what capped it \'\n            \'(the 58-study gold BCE term is small and easy to overfit)\'\n        ) if best_epochs else \'no fold produced a valid best-epoch AUC (too few gold rows per fold)\',\n        \'runtime_seconds\': time.time() - start, \'device\': \'cpu\',\n        \'oof_sha256\': digest_file(oof_path),\n    }\n    write_receipt(out_dir / f\'{arm_name}_receipt.json\', receipt)\n    return frame, receipt\n', encoding='utf-8')
(_src / 'compare_oof.py').write_text('"""Automated, paired comparison of the baseline-arm vs. auxiliary-arm OOF\npredictions against expert gold labels. Same statistical standard as v14\'s\ndiagnostics.py: grouped paired bootstrap, no unpaired comparisons, explicit\nscope caveats rather than a bare number.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\nfrom contract import TARGETS, UID\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef align(frame, ids):\n    if frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(\'Missing or duplicate study IDs\')\n    if set(frame[UID]) != set(ids):\n        raise ValueError(\'Prediction coverage does not exactly match the requested study IDs; \'\n                         \'partial scoring is prohibited\')\n    return frame.set_index(UID).loc[list(ids)].reset_index()\n\n\ndef auc(y, p):\n    valid = np.isfinite(y)\n    y, p = y[valid], p[valid]\n    pos, neg = np.sum(y == 1), np.sum(y == 0)\n    if not pos or not neg:\n        return np.nan\n    return float((rankdata(p)[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))\n\n\ndef aucs(y, p):\n    return np.array([auc(y[:, j], p[:, j]) for j in range(len(TARGETS))])\n\n\ndef compare(labels_path, baseline_oof_path, auxiliary_oof_path, bootstrap=2000, seed=1400):\n    labels = pd.read_csv(labels_path, dtype={UID: str})\n    baseline = pd.read_csv(baseline_oof_path, dtype={UID: str})\n    auxiliary = pd.read_csv(auxiliary_oof_path, dtype={UID: str})\n    ids = labels[UID].tolist()\n    baseline = align(baseline, ids)\n    auxiliary = align(auxiliary, ids)\n    y = labels.set_index(UID).loc[ids, TARGETS].to_numpy(float)\n    b = baseline[TARGETS].to_numpy(float)\n    a = auxiliary[TARGETS].to_numpy(float)\n    if not (np.isin(y, [0.0, 1.0])).all():\n        raise ValueError(\'Labels must be strictly binary for this comparison\')\n\n    ba, aa = aucs(y, b), aucs(y, a)\n    rng = np.random.default_rng(seed)\n    n = len(ids)\n    draws = []\n    for _ in range(bootstrap):\n        idx = rng.integers(0, n, n)\n        d = aucs(y[idx], a[idx]) - aucs(y[idx], b[idx])\n        if np.isfinite(d).all():\n            draws.append(d)\n    if len(draws) < 0.5 * bootstrap:\n        raise ValueError(\'Too many resamples lacked both classes; cohort too small/imbalanced for this comparison\')\n    draws = np.asarray(draws)\n    macro_ci = np.quantile(draws.mean(axis=1), [.025, .975])\n    target_ci = np.quantile(draws, [.025, .975], axis=0)\n\n    return {\n        \'scope\': (\'Paired comparison on the SAME 58-study gold cohort that also selected the \'\n                  \'auxiliary-loss policy (see labels.py). This is exploratory, not independent \'\n                  \'confirmation, and not a hidden-test-set estimate.\'),\n        \'cohort_studies\': n,\n        \'baseline_macro_auc\': float(ba.mean()), \'auxiliary_macro_auc\': float(aa.mean()),\n        \'macro_delta\': float((aa - ba).mean()), \'macro_delta_ci95\': macro_ci.tolist(),\n        \'bootstrap_seed\': seed, \'valid_replicates\': len(draws), \'requested_replicates\': bootstrap,\n        \'targets\': {t: {\'baseline_auc\': float(ba[j]), \'auxiliary_auc\': float(aa[j]),\n                        \'delta\': float(aa[j] - ba[j]), \'delta_ci95\': target_ci[:, j].tolist()}\n                   for j, t in enumerate(TARGETS)},\n        \'input_sha256\': {\'labels\': digest(labels_path), \'baseline_oof\': digest(baseline_oof_path),\n                         \'auxiliary_oof\': digest(auxiliary_oof_path)},\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--labels\', type=Path, required=True)\n    p.add_argument(\'--baseline-oof\', type=Path, required=True)\n    p.add_argument(\'--auxiliary-oof\', type=Path, required=True)\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    p.add_argument(\'--seed\', type=int, default=1400)\n    p.add_argument(\'--out\', type=Path, required=True)\n    args = p.parse_args()\n    result = compare(args.labels, args.baseline_oof, args.auxiliary_oof, args.bootstrap, args.seed)\n    with args.out.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(result, f, indent=2)\n        f.write(\'\\n\')\n    print(f"macro delta {result[\'macro_delta\']:+.6f}, CI95 {result[\'macro_delta_ci95\']}")\n    for t in TARGETS:\n        d = result[\'targets\'][t]\n        print(f"  {t:18s} base={d[\'baseline_auc\']:.4f} aux={d[\'auxiliary_auc\']:.4f} "\n              f"delta={d[\'delta\']:+.4f} ci95={d[\'delta_ci95\']}")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(_src / 'family_c_gate.py').write_text('"""Family C reproducibility gate: only proceed to Family C once Family A shows a\nreproducible, same-direction gain across two independently seeded training runs.\nThis is a project-management gate, not statistical proof (the plan is explicit\nabout that distinction) -- with a 58-study cohort, requiring the bootstrap CI\nlower bound to clear zero on both seeds would almost never pass regardless of\nwhether the underlying effect is real, so the gate checks sign- and\nmagnitude-consistency across seeds instead of a per-seed significance test.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef evaluate_gate(seed_reports, min_macro_gain=0.0):\n    """seed_reports: list of >=2 compare_oof.py-style report dicts, one per\n    training seed, each comparing the same two arms. PASS requires every seed\'s\n    macro_delta to exceed min_macro_gain AND all seeds to agree in sign."""\n    if len(seed_reports) < 2:\n        raise ValueError(\'Need at least two seeds to evaluate reproducibility\')\n    deltas = [r[\'macro_delta\'] for r in seed_reports]\n    signs = {1 if d > 0 else (-1 if d < 0 else 0) for d in deltas}\n    all_above_threshold = all(d > min_macro_gain for d in deltas)\n    consistent_sign = len(signs) == 1 and 0 not in signs\n    passed = all_above_threshold and consistent_sign\n\n    per_target_agreement = {}\n    targets = seed_reports[0][\'targets\'].keys()\n    for t in targets:\n        target_deltas = [r[\'targets\'][t][\'delta\'] for r in seed_reports]\n        target_signs = {1 if d > 0 else (-1 if d < 0 else 0) for d in target_deltas}\n        per_target_agreement[t] = {\n            \'deltas_by_seed\': target_deltas,\n            \'sign_agreement\': len(target_signs) == 1 and 0 not in target_signs,\n        }\n    agreeing_targets = sum(v[\'sign_agreement\'] for v in per_target_agreement.values())\n\n    return {\n        \'decision\': \'PASS_PROCEED_TO_FAMILY_C\' if passed else \'FAIL_DO_NOT_START_FAMILY_C\',\n        \'reason\': (\'all seeds exceed the minimum gain with consistent sign\' if passed else\n                   \'macro deltas disagree in sign or direction across seeds\' if not consistent_sign else\n                   f\'not every seed exceeded the minimum gain of {min_macro_gain}\'),\n        \'macro_deltas_by_seed\': deltas,\n        \'min_macro_gain_required\': min_macro_gain,\n        \'targets_agreeing_in_sign\': agreeing_targets, \'targets_total\': len(per_target_agreement),\n        \'per_target\': per_target_agreement,\n        \'caveat\': (\'Project-management gate, not statistical proof: 58 studies do not support a \'\n                  \'per-seed significance requirement. A PASS means the observed direction was \'\n                  \'reproduced, not that the effect is confirmed at conventional significance.\'),\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--seed-reports\', type=Path, nargs=\'+\', required=True,\n                   help=\'Two or more compare_oof.py JSON outputs, one per training seed\')\n    p.add_argument(\'--min-macro-gain\', type=float, default=0.0)\n    p.add_argument(\'--out\', type=Path, required=True)\n    args = p.parse_args()\n    reports = [json.loads(path.read_text(encoding=\'utf-8\')) for path in args.seed_reports]\n    result = evaluate_gate(reports, args.min_macro_gain)\n    with args.out.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(result, f, indent=2)\n        f.write(\'\\n\')\n    print(result[\'decision\'])\n    print(result[\'reason\'])\n    print(f"targets agreeing in sign: {result[\'targets_agreeing_in_sign\']}/{result[\'targets_total\']}")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
sys.path.insert(0, '/kaggle/working/family_a_src')
print('family_a modules written to', _src)

In [ ]:

import json
import pandas as pd
from contract import TARGETS, UID
from labels import zero_policy
from engine import run_arm
from model import GeneralistModel, Dinov2Encoder

def make_model():
    return GeneralistModel(Dinov2Encoder(find_dinov2('small'), unfreeze_last=2))

pool_cache = Path('/kaggle/working/family_a_pool_cache')
gold_cache = Path('/kaggle/working/family_a_gold_cache')
gold_labels = pd.read_csv('/kaggle/working/family_a_gold_prep/gold_labels.csv', dtype={UID: str})
pool_ids = pd.read_csv(pool_cache / 'cached_ids.csv', dtype=str)[UID].tolist()
gold_ids = pd.read_csv(gold_cache / 'cached_ids.csv', dtype=str)[UID].tolist()
ids = gold_ids + pool_ids  # gold first so gold rows spread across folds deterministically

# Every id must be readable from ONE cache directory; symlink pool+gold into one place.
import os
combined_cache = Path('/kaggle/working/family_a_combined_cache')
combined_cache.mkdir(parents=True, exist_ok=True)
for src_dir, id_list in ((gold_cache, gold_ids), (pool_cache, pool_ids)):
    for uid in id_list:
        dst = combined_cache / f'{uid}.npz'
        if not dst.exists():
            os.symlink((src_dir / f'{uid}.npz').resolve(), dst)

report_source = pd.read_csv('/home/zha76451/Projects/rsna-knee-abnormality/v14' + '/external_labels/llm_labels_v2.csv', dtype={UID: str})
auxiliary_policy = json.loads('{"ACL": {"weight": 0.3, "use_aux": true, "reason": "gate passed: AUC CI lower=0.974, skill=0.592"}, "MCL": {"weight": 0.05500476190476211, "use_aux": true, "reason": "gate passed: AUC CI lower=0.888, skill=0.110, weight halved (small sample)"}, "Medial Meniscus": {"weight": 0.3, "use_aux": true, "reason": "gate passed: AUC CI lower=0.909, skill=0.482"}, "Lateral Meniscus": {"weight": 0.23670454545454567, "use_aux": true, "reason": "gate passed: AUC CI lower=0.741, skill=0.237"}, "Medial OA": {"weight": 0.3, "use_aux": true, "reason": "gate passed: AUC CI lower=0.839, skill=0.428"}, "Lateral OA": {"weight": 0.0, "use_aux": false, "reason": "gate failed: AUC CI lower=0.605, skill=-0.122"}, "PF OA": {"weight": 0.3, "use_aux": true, "reason": "gate passed: AUC CI lower=0.866, skill=0.392"}, "Effusion": {"weight": 0.13164952380952388, "use_aux": true, "reason": "gate passed: AUC CI lower=0.721, skill=0.132"}, "Synovitis": {"weight": 0.0, "use_aux": false, "reason": "gate failed: AUC CI lower=0.685, skill=-0.031"}, "Baker\'s": {"weight": 0.3, "use_aux": true, "reason": "gate passed: AUC CI lower=0.815, skill=0.499"}, "Contusion": {"weight": 0.0, "use_aux": false, "reason": "gate failed: AUC CI lower=0.666, skill=-0.130"}, "Fracture": {"weight": 0.0042843137254900965, "use_aux": true, "reason": "gate passed: AUC CI lower=0.616, skill=0.004"}}')

out_dir = Path('/kaggle/working/family_a_run4')
for seed_value in [1400, 1400 + 1]:
    for arm_name, policy in (('baseline', zero_policy()), ('auxiliary', auxiliary_policy)):
        arm_dir = out_dir / f'seed{seed_value}' / arm_name
        run_arm(arm_name, combined_cache, ids, gold_ids, gold_labels, report_source, policy,
               arm_dir, make_model, k=5, epochs=10, seed=seed_value, patience=4)
        print(f'seed {seed_value} {arm_name} done -> {arm_dir}')
print('DONE training both arms, both seeds. Run compare_oof.py and family_c_gate.py next.')
